# Phase 2 — Negative Review Classification

This is the team's canonical classification notebook. It develops an interpretable Logistic Regression model for predicting which **late-delivered orders** will receive a **negative review (1–2 stars)**. Reusable validation and pipeline code lives in `src/`; this notebook keeps the analytical reasoning, audits, comparisons and conclusions visible for the IT5006 submission.

No model results are stored yet. Run and review the analysis as a team before adding outputs or conclusions.

## 1. Problem & Data

### Problem, stakeholder and prediction point

The stakeholder is the customer-experience team. The decision is which customers to prioritise for recovery or outreach after their orders have already arrived late. The prediction is made **immediately after the late delivery is completed and before the customer's review is known**.

Among late deliveries, the retrospective target is 1 for review scores 1–2 and 0 for scores 3–5. Review identifiers, text and timestamps are outcome-side information and cannot be predictors. Delivery performance is available because the prediction is made after delivery.

Phase 1 already produced cleaned analytical CSVs. Phase 2 consumes `order_level.csv` and `item_level.csv`; it does not repeat general cleaning or use the dashboard Parquet.

```mermaid
flowchart TD
    P1[Phase 1 processed data] --> BUILD[Build late-delivery<br/>classification dataset]
    BUILD --> SPLIT{Protected stratified<br/>train / test split}

    SPLIT -->|Training data only| MODELS[Dummy baseline +<br/>Logistic Regression]
    MODELS --> CV[Stratified cross-validation<br/>and OOF probabilities]
    CV --> DECIDE[Model and threshold decision]

    SPLIT -->|Hold untouched| TEST[(Protected test set)]
    TEST -.->|Open only after decision| FINAL[Final evaluation]
    DECIDE --> FINAL

    FINAL --> INTERPRET[Interpretation and conclusion]
    INTERPRET --> SAVE[Save pipeline + threshold<br/>for Phase 3]
```

### Team collaboration

Teammates can divide the late-delivery cohort/feature audit, model development, and evaluation/interpretation work, including supporting functions in `src/`. Avoid editing this `.ipynb` simultaneously; use branches and one notebook integrator when combining changes.

### Setup and Phase 1 inputs

**Inputs:** `data/processed/order_level.csv` at one row per order and `item_level.csv` at one row per order item.

**Output of dataset construction:** one row per late, delivered, reviewed order containing trace-only `order_id`, the explicit candidate predictors, and `is_negative_review`. The detailed contract and validations are in the `build_classification_dataset()` docstring.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
    cross_val_predict,
    train_test_split,
)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.classification_data import (
    CATEGORICAL_FEATURES,
    MODEL_FEATURES,
    NUMERIC_FEATURES,
    TARGET_COLUMN,
    build_classification_dataset,
    review_timing_audit,
    split_features_target,
)
from src.classification_evaluation import (
    best_threshold_by_metric,
    classification_metrics,
    threshold_table,
)
from src.classification_model import (
    build_dummy_pipeline,
    build_logistic_pipeline,
    predict_from_artifact,
    save_inference_artifact,
)

RANDOM_SEED = 42
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
order_level = pd.read_csv(PROCESSED_DIR / "order_level.csv")
item_level = pd.read_csv(PROCESSED_DIR / "item_level.csv")
print(f"order_level: {order_level.shape}; item_level: {item_level.shape}")

### Construct and audit the modelling dataset

Business-specific item aggregation is deterministic and occurs before modelling. Learned imputation, scaling and encoding remain inside the sklearn Pipeline.

Review timing is reported as a data-quality check rather than used as an eligibility filter or predictor. Late-order eligibility reuses Phase 1 `late_delivery_flag`; no new lateness definition is introduced.

In [ ]:
classification_data = build_classification_dataset(order_level, item_level)
assert classification_data["order_id"].is_unique

column_audit = pd.DataFrame({
    "dtype": classification_data.dtypes.astype(str),
    "missing_n": classification_data.isna().sum(),
    "missing_pct": classification_data.isna().mean(),
    "unique_n": classification_data.nunique(dropna=False),
})
print(f"classification dataset: {classification_data.shape}")
display(column_audit)
display(classification_data[TARGET_COLUMN].value_counts(dropna=False).rename("n").to_frame())
display(classification_data[NUMERIC_FEATURES].describe().T)
display(review_timing_audit(order_level).to_frame())

### Initial candidate feature and leakage audit

The following is the **initial candidate feature set**. The team should review it before modelling. `MODEL_FEATURES` is an explicit allow-list; the notebook never defines features as “all columns except target.”

| Feature | Source | Treatment | Available at prediction point? | Model input? | Reason |
| --- | --- | --- | --- | --- | --- |
| `review_score` | order/review | Target only | No | Target only | Constructs the retrospective target |
| Review ID, comments and review timestamps | review | Excluded leakage | No | No | Outcome-side information |
| Order, customer, product and seller IDs | processed tables | Excluded identifier | Yes | No | Trace/join keys; avoid entity memorisation |
| `order_status` | order | Excluded constant/non-useful | Yes | No | Population is restricted to delivered orders |
| `late_delivery_flag` | order | Cohort filter; excluded constant | Yes | No | Every modelling row is late |
| Raw lifecycle timestamps | order | Derived then excluded | Varies | No | Replaced by reviewed durations/temporal fields |
| Delivery days, estimated days and `days_early` | order | Derived | Yes | Yes | Phase 1 convention is retained: negative `days_early` means late |
| Item/value/freight/seller counts and freight share | order | Derived | Yes | Yes | Order composition and value |
| Payment totals/count/installments | payment | Derived | Yes | Yes | Known before delivery |
| Customer/seller state and same-state fields | order/items | Direct/derived | Yes | Yes | Coarse geography without entity IDs |
| Purchase month and weekday | order | Derived | Yes | Yes | Seasonal context |
| Primary product category | item/product | Derived | Yes | Yes | Deterministic highest-value category |
| Mean product weight, volume and photo count | item/product | Derived | Yes | Yes | Order-grain product characteristics |
| Name/description lengths | product | Excluded non-useful | Yes | No | Not selected for the initial model |

In [ ]:
feature_audit = pd.DataFrame({
    "feature": MODEL_FEATURES,
    "type": ["numeric" if name in NUMERIC_FEATURES else "categorical" for name in MODEL_FEATURES],
    "candidate_model_input": True,
    "available_after_delivery": True,
})
display(feature_audit)
assert not any(name.startswith("review_") for name in MODEL_FEATURES)
assert "order_id" not in MODEL_FEATURES

### Protected stratified split

The test set is created once and then left untouched while features, Logistic Regression settings and the operating threshold are considered using training data only.

In [ ]:
X, y = split_features_target(classification_data)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_SEED,
)
print(f"training rows: {len(X_train):,}; protected test rows: {len(X_test):,}")

## 2. Model Development

### Dummy baseline and Logistic Regression

`DummyClassifier(strategy="prior")` is a no-skill reference: it learns the class prior, not relationships between predictors and the target. It uses the same preprocessing pipeline only for interface consistency.

The Logistic Regression pipeline learns numeric imputation and scaling plus categorical imputation and one-hot encoding inside each training fold, preventing preprocessing leakage.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
scoring = {
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "balanced_accuracy": "balanced_accuracy",
    "pr_auc": "average_precision",
    "roc_auc": "roc_auc",
}

dummy_pipeline = build_dummy_pipeline(strategy="prior")
logistic_pipeline = build_logistic_pipeline(random_state=RANDOM_SEED)

# Run during the modelling analysis and retain generated fold-level results.
# dummy_cv = pd.DataFrame(cross_validate(dummy_pipeline, X_train, y_train, cv=cv, scoring=scoring))
# logistic_cv = pd.DataFrame(cross_validate(logistic_pipeline, X_train, y_train, cv=cv, scoring=scoring))
# display(dummy_cv)
# display(logistic_cv)

### Controlled Logistic Regression refinement

Keep refinement within the Logistic Regression family. A small comparison may consider class weighting or regularisation strength on identical stratified folds. Record the comparison and choose a configuration using training evidence rather than protected test performance.

In [ ]:
weighted_logistic_pipeline = build_logistic_pipeline(
    random_state=RANDOM_SEED,
    class_weight="balanced",
)

# Add a small, justified comparison here and assign the reviewed configuration.
selected_pipeline = clone(logistic_pipeline)

## 3. Evaluation

### OOF threshold analysis

Generate out-of-fold probabilities from the selected training-only configuration. Inspect precision, recall, F1 and balanced accuracy across thresholds. `best_threshold_by_metric()` may identify a statistical candidate, but the operating threshold remains a team decision based on the cost of missed negative reviews versus unnecessary outreach.

In [ ]:
# Run after the Logistic Regression configuration is agreed.
# oof_probability = cross_val_predict(
#     selected_pipeline, X_train, y_train, cv=cv, method="predict_proba"
# )[:, 1]
# oof_thresholds = threshold_table(y_train, oof_probability)
# display(oof_thresholds)
# f1_candidate = best_threshold_by_metric(oof_thresholds, metric="f1")
# selected_threshold = ...  # Document the stakeholder-based team decision.

### Final protected-test evaluation

After the model configuration and threshold are frozen, fit on all training rows and open the test set once. Report a confusion matrix and imbalance-aware metrics including precision, recall, F1, balanced accuracy and PR-AUC. Do not return to model or threshold selection after seeing these results.

In [ ]:
# Run once after the model and threshold decision is frozen.
# final_pipeline = clone(selected_pipeline).fit(X_train, y_train)
# test_probability = final_pipeline.predict_proba(X_test)[:, 1]
# test_metrics = classification_metrics(y_test, test_probability, threshold=selected_threshold)
# display(pd.Series(test_metrics, name="protected_test"))

## 4. Interpretation & Conclusion

### Coefficients, odds ratios and percentage change in odds

The model outcome is `is_negative_review = 1`. Read each row using the course sequence:

**Coefficient → Odds Ratio → % Change in Odds → Interpretation**

- `Odds Ratio = exp(Coefficient)`.
- `% Change in Odds = (Odds Ratio - 1) × 100`.
- A positive coefficient and odds ratio above 1 are associated with **higher odds of a negative review**.
- A negative coefficient and odds ratio below 1 are associated with **lower odds of a negative review**.
- Numeric predictors are standardised inside the Pipeline, so their coefficients describe a **+1 standard deviation increase**, holding the other model variables fixed.

The current `OneHotEncoder(handle_unknown="ignore")` uses its default `drop=None`. It creates a column for every observed category, so there is **no omitted reference category**. An individual categorical odds ratio is therefore tied to the model's regularised intercept/encoding parameterisation; it should not be described as “versus the reference category.” A direct comparison between categories A and B uses `exp(coefficient_A - coefficient_B)`. The team should name a meaningful comparison explicitly if categorical effects are discussed in the report.

The generated interpretation text describes statistical associations, not causes. Business implications must be written by the team after reviewing the fitted results and relevant Phase 1 evidence.

In [ ]:
# Run after the final Logistic Regression fit.
# feature_names = final_pipeline.named_steps["preprocessor"].get_feature_names_out()
# coefficients = final_pipeline.named_steps["classifier"].coef_[0]
#
# coefficient_table = pd.DataFrame({
#     "Feature": feature_names,
#     "Coefficient": coefficients,
# })
# coefficient_table["Odds Ratio"] = np.exp(coefficient_table["Coefficient"])
# coefficient_table["% Change in Odds"] = (
#     coefficient_table["Odds Ratio"] - 1
# ) * 100
#
# def describe_association(row):
#     direction = "higher" if row["Coefficient"] > 0 else "lower"
#     magnitude = abs(row["% Change in Odds"])
#     if row["Feature"] in NUMERIC_FEATURES:
#         comparison = "a +1 standard deviation increase"
#     else:
#         comparison = "this encoded category (no omitted reference category)"
#     return (
#         f"{comparison} is associated with {magnitude:.1f}% {direction} odds "
#         "of a negative review, holding other model variables fixed."
#     )
#
# coefficient_table["Interpretation"] = coefficient_table.apply(
#     describe_association, axis=1
# )
# coefficient_table = coefficient_table.sort_values("Coefficient")
# display(coefficient_table)
#
# # Concise report view: strongest associations in each direction, not every dummy column.
# N_ASSOCIATIONS = 5
# strongest_associations = pd.concat([
#     coefficient_table.nsmallest(N_ASSOCIATIONS, "Coefficient"),
#     coefficient_table.nlargest(N_ASSOCIATIONS, "Coefficient"),
# ]).drop_duplicates("Feature")
# display(strongest_associations.sort_values("Coefficient"))

### Error analysis and conclusion

Compare false negatives and false positives across useful cohorts such as state, category, lateness severity and order value, always showing cohort sizes. Use this analysis to describe limitations and future checks—not to tune against the protected test set.

Conclude whether Logistic Regression is useful for the stated customer-experience decision by comparing it with the Dummy baseline, the agreed success criteria, cross-validation variability and error costs. A fitted model does not have to be accepted.

In [ ]:
# Construct after final evaluation.
# test_errors = X_test.copy()
# test_errors["actual"] = y_test
# test_errors["probability"] = test_probability
# test_errors["predicted"] = (test_probability >= selected_threshold).astype(int)
# display(test_errors.head())

## 5. Save Model for Phase 3

Save only the reviewed fitted pipeline and training-selected threshold, together with provenance metadata. Phase 3 should consume this interface rather than reproduce training transformations. Reloading and scoring one held-out-shaped row provides an inference-interface smoke test; it is not a model-quality test.

In [ ]:
ARTIFACT_PATH = PROJECT_ROOT / "deployment" / "negative_review_logistic.joblib"

# Run only after the final model decision.
# save_inference_artifact(
#     final_pipeline,
#     str(ARTIFACT_PATH),
#     threshold=selected_threshold,
#     metadata={
#         "git_commit": "RECORD_COMMIT",
#         "training_data_contract": "Phase 1 order_level.csv + item_level.csv",
#     },
# )
#
# import joblib
# artifact = joblib.load(ARTIFACT_PATH)
# display(predict_from_artifact(artifact, X_test.iloc[[0]]))